# MCP (Model Context Protocol) — Interactive Guide

Welcome to the **MCP module** of the HFA AI Training. This notebook is your interactive companion for understanding MCP — the open standard that lets AI models connect to external tools and data sources.

> **Think of MCP as a universal adapter.** Instead of building custom integrations for every tool an AI needs, MCP provides one standard protocol for AI to interact with the outside world.

---

## What is MCP?

**Model Context Protocol (MCP)** is an open standard created by Anthropic for connecting AI models to external tools and data sources.

Before MCP, if you wanted Claude to check your database, you had to build a custom integration. If you also wanted it to search your files, that was another separate custom build. MCP says: *"Let's create one protocol that works for everything."*

### The Key Idea

MCP lets you go from an AI that **talks** about doing things to an AI that **actually does** things.

### The Three Players

| Component | What It Is | Example |
|-----------|-----------|--------|
| **MCP Server** | A program that exposes tools, resources, and prompts | A Python script that can search listings |
| **MCP Client** | An AI-powered app that connects to servers | Claude Desktop, Claude Code, Cursor |
| **The AI Model** | The brain that decides what to do | Claude, GPT, etc. |

### How It Works

```
User asks a question
        |
        v
  Claude (Model)     --> "I need to search listings to answer this"
        |
        v
   MCP Client        --> Sends tool call to the right server
        |
        v
   MCP Server        --> Executes the tool, returns results
        |
        v
  Claude (Model)     --> Uses results to form a response
```

MCP servers communicate using **stdio** (standard input/output) for local servers, or **HTTP (SSE)** for remote/shared servers.

---

## Setup: Install the MCP Package

Before working with MCP servers, you need the `mcp` Python package installed.

In [ ]:
!pip install mcp

---

## Exploring the Simple MCP Server

MCP servers are standalone Python scripts — they run as their own process and communicate with AI clients over stdio. That means we **cannot** run an MCP server inside a Jupyter notebook, but we **can** read the code and study it here.

The file `simple_mcp_server.py` defines a basic MCP server with three tools:

1. **`get_weather`** — Returns simulated weather for a Hawaii location
2. **`search_listings`** — Returns simulated real estate listings
3. **`calculate_mortgage`** — Calculates monthly mortgage payments

Let's read the full source code:

In [ ]:
with open('simple_mcp_server.py') as f:
    print(f.read())

### Key Things to Notice

1. **`Server("real-estate-tools")`** — Creates the MCP server with a name that clients will see.
2. **`@server.list_tools()`** — This decorator defines *what tools are available*. The AI model reads the tool names, descriptions, and input schemas to decide when and how to call them.
3. **`@server.call_tool()`** — This decorator handles the *actual execution* when the AI decides to call a tool.
4. **`stdio_server()`** — The server uses stdio transport, meaning Claude Desktop launches it as a subprocess and communicates via stdin/stdout.

### How to Run the Server

To test the server standalone (for development):

```bash
python simple_mcp_server.py
```

In practice, you do **not** run MCP servers manually. Instead, you configure Claude Desktop to launch them automatically (see the next section).

---

## Configuring Claude Desktop

To use an MCP server with Claude Desktop, you edit a JSON configuration file that tells Claude Desktop which servers to launch.

The config file lives at:
- **macOS:** `~/Library/Application Support/Claude/claude_desktop_config.json`
- **Windows:** `%APPDATA%\Claude\claude_desktop_config.json`

Let's look at the example configuration included in this module:

In [ ]:
with open('claude_desktop_config_example.json') as f:
    print(f.read())

### Configuration Patterns Shown Above

| Pattern | Description |
|---------|-------------|
| **Simple local server** | `"command": "python"` + `"args": ["/path/to/server.py"]` — the most common pattern |
| **With environment variables** | Add `"env": {"API_KEY": "..."}` to pass secrets without hardcoding them |
| **Node.js server** | Use `"command": "npx"` for npm-based MCP servers |
| **Using `uv`** | Use the `uv` tool for automatic dependency management |
| **Docker** | Run MCP servers inside containers for isolation |
| **Remote SSE** | Connect to a shared server running over HTTP |

After saving your config, **completely quit and reopen** Claude Desktop. The servers start automatically, and you will see a hammer/tools icon in new conversations.

---

## Pros and Cons of MCP

### Pros

| Advantage | Why It Matters |
|-----------|---------------|
| **Standardized protocol** | Build once, works with Claude Desktop, Claude Code, Cursor, and more |
| **Real system access** | Agents can query databases, call APIs, read/write files — do real work |
| **Composable** | Mix and match servers. Need listings + weather + email? Add three servers |
| **Secure by design** | You control exactly which tools the agent can access. Nothing hidden |
| **Open source** | The protocol is open — anyone can build servers and clients |
| **Growing ecosystem** | Thousands of pre-built MCP servers already available |

### Cons

| Drawback | What to Watch For |
|----------|------------------|
| **Still evolving** | The protocol is under active development — things may change |
| **Setup complexity** | Configuring servers requires some technical comfort (JSON config, Python, etc.) |
| **Security power** | The agent has real power — a misconfigured tool could modify real data |
| **Debugging** | When something goes wrong, tracing the issue across client/server can be tricky |
| **Latency** | Each tool call adds time — complex workflows with many calls can feel slow |
| **Context window usage** | Tool definitions and results consume tokens from the model's context |

---

## Advanced: MCP Server with Resources

The second server in this module — `mcp_with_resources.py` — demonstrates a more advanced pattern: exposing both **tools** and **resources**.

Let's read the full source:

In [ ]:
with open('mcp_with_resources.py') as f:
    print(f.read())

---

## Tools vs Resources

This is one of the most important concepts in MCP. Understanding the difference helps you design better servers.

| | **Tools** | **Resources** |
|---|-----------|---------------|
| **What they are** | Actions the agent can TAKE | Data the agent can READ |
| **Think of them as** | **Verbs** — they DO things | **Nouns** — they ARE things |
| **Examples** | Search, calculate, send, update, delete | Documents, records, configs, files |
| **State changes?** | Yes — tools can modify data | No — resources are read-only |
| **Identified by** | A name (e.g., `search_documents`) | A URI (e.g., `docs://buyer-guide`) |
| **When to use** | When the agent needs to *do* something | When the agent needs to *know* something |

### In the `mcp_with_resources.py` example:

**Resources** (data to read):
- `docs://buyer-guide` — Hawaii Home Buyer's Guide
- `docs://seller-checklist` — Seller's Pre-Listing Checklist
- `docs://market-report-2026-q1` — Oahu Market Report Q1 2026

**Tools** (actions to take):
- `search_documents` — Search across all documents by keyword
- `add_document` — Add a new document to the store
- `get_document_stats` — Get statistics about the document library

Notice how `search_documents` is a **tool** (not a resource) because it performs an action — filtering and ranking results based on a query. Meanwhile, the documents themselves are **resources** that can be read directly by URI.

---

## When to Use MCP

### Use MCP When...

- You want an AI agent to **DO things**, not just talk about them
- You are **automating workflows** that touch external systems (databases, APIs, file systems)
- You are building **internal tools** powered by AI for your team
- You want to give Claude access to **your specific data** (company docs, CRM, listings)
- You need a **standardized approach** that works across multiple AI clients
- You want **composable capabilities** — add/remove tools without changing the AI

### Do NOT Use MCP When...

- You just need **simple Q&A** that does not require external data
- Your **security constraints** are extremely tight and you cannot allow any tool execution
- The task is a **one-off** that is faster to do manually than to set up a server
- You need **real-time streaming** of large datasets (MCP is better for request/response patterns)
- **Pre-built integrations** already exist (e.g., Claude's built-in web search)

### Key Takeaways

1. **MCP is the USB-C of AI tools** — one standard connector for everything
2. **Agentifying = giving AI control** of multi-step workflows via tools
3. **Servers expose tools and resources**, clients connect the AI to them
4. **You control the power** — only expose what the agent should access
5. **Use it when the AI needs to DO things** — not just when it needs to talk

---

### Further Reading

- [MCP Official Documentation](https://modelcontextprotocol.io)
- [MCP GitHub Repository](https://github.com/modelcontextprotocol)
- [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk)
- [Claude Desktop MCP Setup Guide](https://docs.anthropic.com/en/docs/agents-and-tools/mcp)
- [Awesome MCP Servers](https://github.com/punkpeye/awesome-mcp-servers) — community directory of pre-built servers